In [1]:
from __future__ import annotations

import argparse
from datasets import load_dataset, load_from_disk
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from infer_min_seg import segment_document
from miniseg_core import find_best_checkpoint, load_checkpoint, simple_sentence_split

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_text(file_path: str | None, inline_text: str | None) -> str:
    if inline_text and inline_text.strip():
        return inline_text.strip()
    if file_path:
        return Path(file_path).read_text(encoding="utf-8")
    raise ValueError("Pass either --file or --text.")


def load_miniseg_model(model_dir: str, checkpoint: str | None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if checkpoint:
        ckpt = Path(checkpoint)
    else:
        model_dir_path = Path(model_dir)
        if model_dir_path.is_file():
            ckpt = model_dir_path
        else:
            ckpt = find_best_checkpoint(model_dir_path)
    if ckpt is None:
        raise FileNotFoundError(f"No checkpoint found under: {model_dir}")
    model, tokenizer, threshold, config, metrics = load_checkpoint(ckpt, device=device)
    return model, tokenizer, threshold, config, metrics, ckpt


def load_led_hier_model(
    base_model_name: str,
    hier_dir: str,
    device: str | None = None,
):
    from inference import load_model as load_led_model

    adapter = "hier"
    model, tokenizer = load_led_model(
        base_model_name=base_model_name,
        adapter=adapter,
        peft_dir="",
        hier_dir=hier_dir,
        device=device or ("cuda" if torch.cuda.is_available() else "cpu"),
    )
    return model, tokenizer


def load_title_t5_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, tokenizer, device


def normalize_text(text: str) -> str:
    return " ".join((text or "").strip().split())

In [3]:
def generate_led_summary(
    text: str,
    model,
    tokenizer,
    min_length: int,
    max_target: int,
) -> str:
    from inference import generate_summary as led_generate_summary

    return led_generate_summary(
        text,
        model=model,
        tokenizer=tokenizer,
        adapter="hier",
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        min_length=min_length,
        max_target=max_target,
    ).strip()

def generate_title_t5(
    summary: str,
    chunk_text: str,
    model,
    tokenizer,
    device: torch.device,
    max_new_tokens: int,
    num_beams: int,
) -> str:
    context = " ".join(simple_sentence_split(chunk_text)[:2]).strip()
    if(chunk_text is not None):
        prompt = (
            "Generate a concise topic title (5 to 10 words) in third person.\n"
            f"Summary: {summary}\n"
            f"Context: {context}\n"
            "Title:"
        )
    else:
        prompt = (
            "Generate a concise topic title (5 to 10 words) in third person.\n"
            f"Summary: {summary}\n"
            "Title:"
        )

    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return normalize_text(tokenizer.decode(output_ids[0], skip_special_tokens=True)) or "Topic Overview"

In [4]:
def format_text_output(global_summary: str, topics: List[Dict[str, Any]]) -> str:
    lines = ["Global summary:", "", global_summary, "", "Topics:", ""]
    for topic in topics:
        lines.append(f"[{topic['topic_index']}] {topic['title']}")
        lines.append(topic["summary"])
        lines.append("")
    return "\n".join(lines).strip() + "\n"

In [5]:
miniseg_model_dir = r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\models\miniseg"
miniseg_max_sentences = 512
miniseg_max_tokens = 128
min_chunk_sentences = 2
led_base_model_name = "allenai/led-base-16384"
led_hier_dir = r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\models\led-tib-hier\led_hier_adapter_run"
title_model = r"google/flan-t5-small"
led_min_length = 50
led_max_target_topic = 320
title_max_new_tokens = 16
title_num_beams = 4
led_max_target_global = 512

In [6]:
miniseg_model, miniseg_tokenizer, saved_threshold, config, metrics, ckpt = load_miniseg_model(model_dir=miniseg_model_dir, checkpoint=None)

led_model, led_tokenizer = load_led_hier_model(base_model_name=led_base_model_name, hier_dir=led_hier_dir)

title_model, title_tokenizer, title_device = load_title_t5_model(title_model)

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x1801f79a800>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 70bc55f3-24e6-411d-98a3-9b37405a6f88)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json
Retrying in 1s [Retry 1/5].
b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [7]:
ytseg_test = load_from_disk(r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\datasets\ytseg-dataset\test")
ind = 4
file = None #r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\texts\What_was_Euclid_really_doing__Guest_video_by_Ben_Syversen.txt"
text=" ".join(ytseg_test["text_ref"][ind])
if(text is not None):
    print(ytseg_test["target_ts"][ind])

[CSTART] 00:00:00 - Introduction [CEND]
[CSTART] 00:00:45 - Natural Log [CEND]
[CSTART] 00:01:18 - Understanding Growth [CEND]
[CSTART] 00:03:44 - Growth Formula [CEND]
[CSTART] 00:07:38 - What is e? [CEND]
[CSTART] 00:08:28 - Value of e [CEND]


In [8]:
text = load_text(file, text)

segmentation = segment_document(
    text=text,
    model=miniseg_model,
    tokenizer=miniseg_tokenizer,
    threshold=0.8,
    max_sentences=min(miniseg_max_sentences, config.max_sentences),
    max_tokens= miniseg_max_tokens,
    min_chunk_sentences=min_chunk_sentences,
)

print("Number of segments:", len(segmentation["chunks"]))

for idx, chunk in enumerate(segmentation["chunks"], start=1):
    print(
        f"Chunk {idx}: start={chunk['start_sentence']}, "
        f"end={chunk['end_sentence']}, "
        f"boundary_score={chunk.get('boundary_score')}, "
        f"text_preview={chunk['text'][:200]!r}"
    )


Number of segments: 3
Chunk 1: start=0, end=13, boundary_score=0.9325082302093506, text_preview='What is ‘e ’? The two most common logarithms used are log to the base 10 and log to the base e. Why are they the most commonly used logarithms? How do we really understand them? Log to the base 10 is '
Chunk 2: start=14, end=47, boundary_score=0.8195534944534302, text_preview='To get to e, we first need to understand GROWTH! Let’s say a particular thing DOUBLES every time period. On this time line, this is today, and every unit is one time period. Assume you have a dollar w'
Chunk 3: start=48, end=95, boundary_score=0.7368210554122925, text_preview='Let’s see how it really works. We take the example of a dollar growing over one year at a 100 percent growth rate. First, we look at the annual growth! Based on what we saw, at time zero, we would hav'


In [ ]:
topics: List[Dict[str, Any]] = []
for idx, chunk in enumerate(segmentation["chunks"], start=1):
    topic_summary = generate_led_summary(
        text=chunk["text"],
        model=led_model,
        tokenizer=led_tokenizer,
        min_length=led_min_length,
        max_target=led_max_target_topic,
    )
    topic_title = generate_title_t5(
        summary=topic_summary,
        chunk_text=chunk["text"],
        model=title_model,
        tokenizer=title_tokenizer,
        device=title_device,
        max_new_tokens=title_max_new_tokens,
        num_beams=title_num_beams,
    )
    topics.append(
            {
                "topic_index": idx,
                "title": topic_title,
                "summary": topic_summary,
                "start_sentence": chunk["start_sentence"],
                "end_sentence": chunk["end_sentence"],
                "boundary_score": float(chunk["boundary_score"]),
            }
        )

global_summary = generate_led_summary(
    text=text,
    model=led_model,
    tokenizer=led_tokenizer,
    min_length=max(40, led_min_length),
    max_target=led_max_target_global,
)

global_title = generate_title_t5(
    summary=global_summary,
    chunk_text=None,
    model=title_model,
    tokenizer=title_tokenizer,
    device=title_device,
    max_new_tokens=title_max_new_tokens,
    num_beams=title_num_beams,
)

output = format_text_output(global_summary, topics)


stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = "_".join(global_title.split())
out_dir = r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\summaries"
out_path = Path(out_dir) / f"{file_name}_summary.txt"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(output, encoding="utf-8")


print(f"MiniSeg checkpoint: {ckpt}")
print("Title: ", global_title)
print(output)
print(f"Wrote text output to: {out_path}")


Input ids are automatically padded from 257 to 1024 to be a multiple of `config.attention_window`: 1024


Input ids are automatically padded from 495 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 720 to 1024 to be a multiple of `config.attention_window`: 1024
Input ids are automatically padded from 1865 to 2048 to be a multiple of `config.attention_window`: 1024


MiniSeg checkpoint: B:\stacks\Machine_learning_projects\Video_Summarization_Translation\models\miniseg\best.pt
Title:  Quantum growth rate
Global summary:

This talk will give an overview of the exponential growth rate in nature. It will also explain how it is possible to grow over a year at 100% annual growth rates. We will then look at some of the more common exponential growth rates and see how they can be used to calculate infinite money.

Topics:

[1] Log-to-the-base-10 is a natural number
Log-to-the-base-10 is a natural number. It can be used to manage multiples, such as the LN and log-to_base-e. In this paper, we will look at some of the most popular log-arithm in mathematics.

[2] GROWTH!
In this lecture, the professor went on to discuss the nature of growth and how it happens in steps. He then moved on to the topic of Growth in Nature, which he described as “growth in steps” or “continuous growth”.

[3] 100 percent growth rate: how it works
This video shows how a 100% annual g